# Notebook 3: Level 1 - Vector Search + Batch Features

**What changes:**
- Look up the customer profile from `customer_profile` FeatureView
- Filter document retrieval by the customer's `plan_type`
- Include personalized context in the LLM prompt

**Result:** A 'term' plan customer gets term-relevant docs, not pension docs.

**Feast APIs used:** `get_online_features()`, `retrieve_online_documents()` with metadata filter

In [ ]:
import os, sys
REPO_DIR = os.path.expanduser('~/life-insurance-rag')
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'app'))

from feast import FeatureStore
from sentence_transformers import SentenceTransformer

store = FeatureStore(repo_path='feature_repo')
model = SentenceTransformer('all-MiniLM-L6-v2')

## Step 1: Fetch Customer Profile (Batch Feature)

In [ ]:
customer_id = 'POL-1001'  # Priya Sharma - Term plan

profile = store.get_online_features(
    features=[
        'customer_profile:customer_name',
        'customer_profile:plan_type',
        'customer_profile:coverage_amount',
        'customer_profile:premium_frequency',
        'customer_profile:nominee_name',
        'customer_profile:city',
    ],
    entity_rows=[{'customer_id': customer_id}],
).to_dict()

print(f'Customer: {profile["customer_name"][0]}')
print(f'Plan: {profile["plan_type"][0]}')
print(f'Coverage: Rs {profile["coverage_amount"][0]:,.0f}')
print(f'City: {profile["city"][0]}')

## Step 2: Vector Search with Metadata Filter

Only documents matching the customer's plan type are returned.

In [ ]:
query = 'What are the payment options for my premium?'
query_vector = model.encode(query).tolist()
plan_type = profile['plan_type'][0]

results = store.retrieve_online_documents(
    feature='insurance_docs:embedding',
    query=query_vector,
    top_k=5,
)

docs_df = results.to_df()
print(f'Retrieved {len(docs_df)} documents')
print(f'Customer plan: {plan_type}\n')

for i, row in docs_df.iterrows():
    match = 'MATCH' if row.get('plan_type', '') in (plan_type, 'all') else 'skip'
    print(f'[{i+1}] [{match}] {row["title"]} (plan_type={row.get("plan_type", "?")})')
    print(f'    {row["content"][:100]}...\n')

## Step 3: Build Personalized Prompt

In [ ]:
# Filter to relevant docs only
relevant_docs = docs_df[docs_df['plan_type'].isin([plan_type, 'all'])]

profile_text = (
    f"Customer: {profile['customer_name'][0]}\n"
    f"Plan: {profile['plan_type'][0]}\n"
    f"Coverage: Rs {profile['coverage_amount'][0]:,.0f}\n"
    f"Premium: {profile['premium_frequency'][0]}\n"
    f"Nominee: {profile['nominee_name'][0]}\n"
    f"City: {profile['city'][0]}"
)

doc_context = '\n---\n'.join(
    f'{row["title"]}: {row["content"]}'
    for _, row in relevant_docs.iterrows()
)

prompt = f"""You are a helpful life insurance assistant.

CUSTOMER PROFILE:
{profile_text}

DOCUMENTS:
{doc_context}

QUESTION: {query}

Answer personally and accurately for this customer."""

print('--- Personalized Prompt (truncated) ---')
print(prompt[:600])
print('...')

## Comparison: Level 0 vs Level 1

| Aspect | Level 0 | Level 1 |
|--------|---------|----------|
| Search | Pure vector | Vector + metadata filter |
| Context | Documents only | Documents + customer profile |
| Personalization | None | By plan type |
| Answer quality | Generic | Plan-specific |

---
**Next:** Open `04_level2_streaming_hybrid.ipynb` to add live claim status + hybrid search.